In [1]:
import yaml
from pprint import pprint  # optional, for nicer printing

# Path to your file
file_path = "../library/exxon/turbines/ATB5_5_base.yaml"

# Open and load the YAML file
with open(file_path, "r") as f:
    turbine_data = yaml.safe_load(f)

# Print out the top-level keys to see the structure
pprint(turbine_data)

{'blades': {'failures': [{'description': 'minor repair',
                          'level': 2,
                          'materials': 407,
                          'operation_reduction': 0,
                          'scale': 13.783,
                          'service_equipment': 'CTV',
                          'shape': 1,
                          'time': 9},
                         {'description': 'major repair',
                          'level': 4,
                          'materials': 3586,
                          'operation_reduction': 0,
                          'scale': 314.328,
                          'service_equipment': 'CTV',
                          'shape': 1,
                          'time': 21},
                         {'description': 'major replacement',
                          'level': 6,
                          'materials': 215219,
                          'operation_reduction': 0.02,
                          'replacement': True,
                    

In [8]:
import yaml
import pandas as pd
from pathlib import Path

def yaml_to_csv(filepaths):
    """
    Convert one or more turbine YAML files into CSV tables of component-level
    maintenance and failure data.

    Each YAML file is saved as a separate CSV with the same base filename.
    """
    for filepath in filepaths:
        # Convert to Path object
        path = Path(filepath)
        if not path.exists():
            print(f"⚠️ File not found: {filepath}")
            continue

        # Load YAML data
        with open(path, "r") as f:
            turbine_data = yaml.safe_load(f)

        # Build records
        records = []
        for component, data in turbine_data.items():
            if isinstance(data, dict):
                # Failures
                if 'failures' in data:
                    for fail in data['failures']:
                        records.append({
                            'component': component,
                            'type': 'failure',
                            'description': fail.get('description'),
                            'level': fail.get('level'),
                            'materials': fail.get('materials'),
                            'scale': fail.get('scale'),
                            'shape': fail.get('shape'),
                            'time': fail.get('time'),
                        })
                # Maintenance
                if 'maintenance' in data:
                    for maint in data['maintenance']:
                        records.append({
                            'component': component,
                            'type': 'maintenance',
                            'description': maint.get('description'),
                            'level': None,
                            'materials': maint.get('materials'),
                            'scale': None,
                            'shape': None,
                            'time': maint.get('time'),
                        })

        # Convert to DataFrame
        df_turbine = pd.DataFrame(records)

        # Save CSV in same directory, named after YAML file
        output_path = path.with_suffix('.csv')
        df_turbine.to_csv(output_path, index=False)

        print(f"✅ Saved: {output_path} ({len(df_turbine)} rows)")


yaml_to_csv(["../library/exxon/turbines/ATB5_5_base.yaml", "../library/exxon/turbines/ATB3_3_base.yaml", "../library/exxon/turbines/ge1_5_base.yaml", "../library/exxon/substations/lbw_substation_500MW.yaml", "../library/exxon/cables/array_66kv_185mm.yaml", "../library/exxon/cables/export_220kv_1000mm.yaml"])

✅ Saved: ..\library\exxon\turbines\ATB5_5_base.csv (66 rows)
✅ Saved: ..\library\exxon\turbines\ATB3_3_base.csv (66 rows)
✅ Saved: ..\library\exxon\turbines\ge1_5_base.csv (66 rows)
✅ Saved: ..\library\exxon\substations\lbw_substation_500MW.csv (3 rows)
✅ Saved: ..\library\exxon\cables\array_66kv_185mm.csv (0 rows)
✅ Saved: ..\library\exxon\cables\export_220kv_1000mm.csv (0 rows)


In [1]:
import pandas as pd
import yaml

df = pd.read_csv("../library/base_2023/turbines/3_3MW_turbine_data.csv")

# === Step 2: Build the YAML dictionary ===
yaml_dict = {
    "capacity_kw": 3300,
    "capex_kw": 915,
    "power_curve": {
        "file": "ATB3_3_SAM.csv",
        "bin_width": 0.25
    }
}

for component in df["component"].unique():
    comp_df = df[df["component"] == component]
    yaml_dict[component] = {
        "name": component,
        "maintenance": [
            {
                "description": "n/a",
                "time": 0,
                "materials": 0,
                "service_equipment": "RMT",
                "frequency": 0
            }
        ],
        "failures": []
    }
    for _, row in comp_df.iterrows():
        yaml_dict[component]["failures"].append({
            "scale": float(row["scale"]),
            "shape": float(row["shape"]),
            "time": float(row["time"]),
            "materials": float(row["materials"]),
            "service_equipment": "CTV" if row["level"] < 6 else "LCN",
            "operation_reduction": float(row["operation_reduction"]),
            "level": int(row["level"]),
            "description": row["description"],
            **({"replacement": True} if "replacement" in row["description"] else {})
        })

# === Step 3: Write YAML file ===
with open("New_Turbine.yaml", "w") as f:
    yaml.dump(yaml_dict, f, sort_keys=False, width=70)

print("✅ YAML file created successfully: New_Turbine.yaml")

✅ YAML file created successfully: New_Turbine.yaml
